In [0]:
from pyspark.sql.functions import col
from pyspark.sql import functions as F

# Load silver master
silver_path = "/Volumes/workspace/default/olist_raw/silver/"
silver = spark.read.format("delta").load(silver_path + "master")

# Select relevant features for predicting review score
ml_data = silver.select(
    "review_score",
    "price",
    "freight_value",
    "payment_value",
    "payment_installments",
    "delivery_days",
    "delivery_on_time",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
).dropna()

print(f"ML dataset: {ml_data.count()} rows, {len(ml_data.columns)} columns")
ml_data.show(3)

ML dataset: 113915 rows, 11 columns
+------------+-----+-------------+-------------+--------------------+-------------+----------------+----------------+-----------------+-----------------+----------------+
|review_score|price|freight_value|payment_value|payment_installments|delivery_days|delivery_on_time|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+------------+-----+-------------+-------------+--------------------+-------------+----------------+----------------+-----------------+-----------------+----------------+
|           5| 84.9|        15.35|       100.25|                   1|           19|               1|             800|               37|               14|              37|
|           1|22.99|        22.85|        91.68|                   9|           25|               0|             150|               19|                4|              11|
|           4| 38.5|        24.84|        63.34|                   5|           21|               1|         

In [0]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd

# Convert to pandas for sklearn
pdf = ml_data.toPandas()

# Features and target
X = pdf.drop("review_score", axis=1)
y = pdf["review_score"]

# Split into train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {len(X_train)} rows")
print(f"Test set: {len(X_test)} rows")

Training set: 91132 rows
Test set: 22783 rows


In [0]:
# Set experiment name
mlflow.set_experiment("/olist-review-score-prediction")

# Start MLflow run
with mlflow.start_run(run_name="random_forest_v1"):
    
    # Model parameters
    n_estimators = 100
    max_depth = 10
    
    # Log parameters
    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("max_depth", max_depth)
    mlflow.log_param("test_size", 0.2)
    
    # Train model
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )
    model.fit(X_train, y_train)
    
    # Evaluate
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="weighted")
    
    # Log metrics
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("f1_score", f1)
    
    # Save model
    mlflow.sklearn.log_model(model, "random_forest_model")
    
    print(f"Accuracy: {round(accuracy * 100, 2)}%")
    print(f"F1 Score: {round(f1, 4)}")

2026/05/04 21:56:10 INFO mlflow.tracking.fluent: Experiment with name '/olist-review-score-prediction' does not exist. Creating a new experiment.
2026/05/04 21:56:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-95de1f4d-69ff.cloud.databricks.com/ml/experiments/3075892757189823/models/m-a4af54b8b08347cbb50e2ab37e1e336b?o=7474645777982763
2026/05/04 21:56:25 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when logging the model to auto infer the model signature. To manually set the signature, please visit https://www.mlflow.org/docs/3.8.1/ml/model/signatures.html for instructions on setting signature on models.


Accuracy: 60.36%
F1 Score: 0.4808


In [0]:
with mlflow.start_run(run_name="random_forest_v2"):
    
    # Different parameters this time
    n_estimators = 200
    max_depth = 15
    
    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("max_depth", max_depth)
    mlflow.log_param("test_size", 0.2)
    
    model_v2 = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )
    model_v2.fit(X_train, y_train)
    
    y_pred_v2 = model_v2.predict(X_test)
    accuracy_v2 = accuracy_score(y_test, y_pred_v2)
    f1_v2 = f1_score(y_test, y_pred_v2, average="weighted")
    
    mlflow.log_metric("accuracy", accuracy_v2)
    mlflow.log_metric("f1_score", f1_v2)
    mlflow.sklearn.log_model(model_v2, "random_forest_model")
    
    print(f"Accuracy: {round(accuracy_v2 * 100, 2)}%")
    print(f"F1 Score: {round(f1_v2, 4)}")

2026/05/04 21:59:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-95de1f4d-69ff.cloud.databricks.com/ml/experiments/3075892757189823/models/m-2627f131190a41eeb6a745cba2ec3090?o=7474645777982763
2026/05/04 21:59:25 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when logging the model to auto infer the model signature. To manually set the signature, please visit https://www.mlflow.org/docs/3.8.1/ml/model/signatures.html for instructions on setting signature on models.


Accuracy: 62.58%
F1 Score: 0.5221
